# ChromaDB Collection Operations (CRUD)

In any production RAG system, the document store is not static. New documents arrive, existing ones get updated, outdated ones need removal. ChromaDB supports the full **CRUD** (Create, Read, Update, Delete) lifecycle on its collections. 


Some of the available methods are:

- **`add()`**: insert new documents with unique IDs and optional metadata
- **`update()`**: replace the document text and/or metadata for an existing ID. The embedding is automatically recomputed
- **`upsert()`**: insert-or-update: if the ID exists it updates; if not, it inserts. Ideal for idempotent ingestion pipelines
- **`get()`**: retrieve documents by ID or metadata filter (no embedding needed)
- **`delete()`**: remove documents by ID or by metadata filter
- **`query()`**: semantic search: embed a query and find nearest neighbours, optionally filtered by metadata
- **`peek()`** / **`count()`**: inspect collection contents without a query

Below we walk through each operation step by step on a fresh collection

#### Setup

Install the required libraries and chunk documents

In [ ]:
import pandas as pd, numpy as np
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

dir_path = "data/Policy Documents"

loader = DirectoryLoader(
    dir_path,
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

# Recursive character splitting
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = recursive_splitter.split_documents(documents)

### Persistent Client (Data Survives Restarts)



Everything we've done so far uses **in-memory** ChromaDB clients, fast and convenient, but all data is lost when the Python process ends. In a real application you need **persistence**: the ability to embed your corpus once and reuse the stored vectors across sessions without recomputation.

ChromaDB's `PersistentClient` writes all collection data (vectors, documents, metadata) to a local directory using SQLite and Parquet files. On the next startup, it reloads from disk automatically.

For production deployments at scale, ChromaDB also supports a **client-server mode** where the server runs as a separate process (or container) and clients connect over HTTP.

In [8]:
import chromadb

crud_client = chromadb.PersistentClient(path="./data/chroma_persistent_db")      # The path to persist directory, where the embeddings will be stored

## Working with Collections

### Managing Collections for a Client

You can add embeddings or chunks of text directly to chroma using their collection methods. To work with collections, Chroma offers the follwing methods.

* `.create_collection()`
* `.get_or_create_collection()`
* `.list_collections()`
* `.delete_collection()`

You saw that we have an option of defining a custom `embedding_function` parameter. Let's see how to use that

In [ ]:
# import the desired embedding model's function
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction, OpenAIEmbeddingFunction

# Continue to declare the embedding function
# Here, we are using the MiniLM-L6-v2
embedding_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")


# For example, if you wish to use an OpenAI embedding model
# model = "text-embedding-ada-002"
# embedding_fn = OpenAIEmbeddingFunction(api_key=OPENAI_API_KEY, model_name=model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


You already know the `.get_or_create_collection()` method.

In [14]:
kb = crud_client.get_or_create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_fn
)
kb

Collection(name=knowledge_base)

If you now try creating a collection with the same name using `.create_collection()`, an error will be raised

In [15]:
kb = crud_client.create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_fn
)
kb

InternalError: Collection [knowledge_base] already exists

Let's create another collection

In [20]:
coll_2 = crud_client.create_collection(
    name="demo_coll_2",
    metadata={"hnsw:space": "cosine"})
coll_2

Collection(name=demo_coll_2)

We can also list and delete collections

In [21]:
crud_client.list_collections()

[Collection(name=demo_coll_2), Collection(name=knowledge_base)]

In [22]:
crud_client.delete_collection("demo_coll_2")
crud_client.list_collections()

[Collection(name=knowledge_base)]

### Handling Documents in Collections

We can add, update, or delete documents in a collection with IDs and metadata.

`collection.add()` is used to add documents to a collection

In [24]:
texts = [doc.page_content for doc in chunks]
metadatas = [doc.metadata for doc in chunks]

kb.add(
    documents=texts,
    metadatas=metadatas,
    ids=[f"doc_{i}" for i in range(len(texts))]
)

In [30]:
# See how texts and metadata look

texts[0], metadatas[0]

('Part A \n<<Date>> \n<<Policyholder’s Name>>  \n<<Policyholder’s Address>> \n<<Policyholder’s Contact Number>> \n \nDear <<Policyholder’s Name>>,  \n \nSub: Your Policy no. <<  >> \nWe are glad to inform you that your proposal has been accepted and the HDFC Life Easy Health (“Policy”)',
 {'producer': 'Microsoft: Print To PDF',
  'creator': 'PyPDF',
  'creationdate': '2021-11-29T10:03:02+00:00',
  'author': 'ANINDYAA',
  'moddate': '2021-12-09T06:23:28+00:00',
  'title': 'HDFC Life Easy Health - 101N110V03 - Policy Bond (Single Pay)',
  'source': 'data\\Policy Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf',
  'total_pages': 33,
  'page': 0,
  'page_label': '1'})

Using `collection.update()`, we can update the embeddings, metadatas or documents for provided ids. It supports one or many updations, by giving lists for the values to update.

In [27]:
kb.update(ids = ["doc_0"],
          # embeddings= [[...]],  # Embedding to update in place of the original one, having the same dimensions
          documents = ["Hi"],       # text to replace
          metadatas = [{'source': 'edited',
                        'total_pages': 30,
                        'page': 19,
                        'page_label': '20'}]
          )

Let's see how the update operation changed the document. We use the `collection.get()` method for it.

In [29]:
# Get by ID
result = kb.get(ids=["doc_0"])

result

{'ids': ['doc_0'],
 'embeddings': None,
 'documents': ['Hi'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'page': 19,
   'source': 'edited',
   'total_pages': 30,
   'producer': 'Microsoft: Print To PDF',
   'creationdate': '2021-11-29T10:03:02+00:00',
   'title': 'HDFC Life Easy Health - 101N110V03 - Policy Bond (Single Pay)',
   'author': 'ANINDYAA',
   'moddate': '2021-12-09T06:23:28+00:00',
   'creator': 'PyPDF',
   'page_label': '20'}]}

We can see the exact fields that we updated have changed.

Now, just like `client.get_or_create_collection()` for creating collections or just fetching one if it already exists; we have the `collection.upsert()` method for inserting (adding) documents to the collection or just updating them if they already exist.

In [31]:
# Let's try with two docs - one already existing and one new
kb.upsert(
    ids=["001", "doc_0"],
    documents=[
        "RAG = Retrieval-Augmented Generation, the dominant pattern for knowledge-grounded AI.",
        "Bye!"
    ],
    metadatas=[
        {"source": "blog", "topic": "RAG", "version": 3},
        {"source": "second edit"}
    ]
)

In [ ]:
# Get by IDs
result = kb.get(ids=["001", "doc_0"], limit=2)

result

{'ids': ['doc_0', '001'],
 'embeddings': None,
 'documents': ['Bye!',
  'RAG = Retrieval-Augmented Generation, the dominant pattern for knowledge-grounded AI.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'creationdate': '2021-11-29T10:03:02+00:00',
   'author': 'ANINDYAA',
   'page': 19,
   'page_label': '20',
   'total_pages': 30,
   'creator': 'PyPDF',
   'producer': 'Microsoft: Print To PDF',
   'title': 'HDFC Life Easy Health - 101N110V03 - Policy Bond (Single Pay)',
   'moddate': '2021-12-09T06:23:28+00:00',
   'source': 'second edit'},
  {'source': 'blog', 'topic': 'RAG', 'version': 3}]}

We can also use the `collection.get()` method to filter using the metadata. It uses JSON-style document queries that you might have seen for MongoDB queries.

In [33]:
kb.get(where = {"$or": [{"source": "second edit"}, {"topic": "RAG"}]})

{'ids': ['doc_0', '001'],
 'embeddings': None,
 'documents': ['Bye!',
  'RAG = Retrieval-Augmented Generation, the dominant pattern for knowledge-grounded AI.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'total_pages': 30,
   'author': 'ANINDYAA',
   'moddate': '2021-12-09T06:23:28+00:00',
   'producer': 'Microsoft: Print To PDF',
   'creationdate': '2021-11-29T10:03:02+00:00',
   'title': 'HDFC Life Easy Health - 101N110V03 - Policy Bond (Single Pay)',
   'page': 19,
   'source': 'second edit',
   'page_label': '20',
   'creator': 'PyPDF'},
  {'version': 3, 'source': 'blog', 'topic': 'RAG'}]}

Finally, to delete the documents, we use the `collection.delete()` method. It deletes using IDs or other metadata filters.

In [67]:
# deleting using ID
# kb.delete(ids = ["001", "doc_0"])

# deleting using filters
kb.delete(where = {"source": "second edit"}, where_document = {"$contains": "Retrieval-Augmented Generation"})

In [68]:
# kb.get(ids=["001", "doc_0"])

kb.get(where = {"source": "second edit"}, where_document = {"$contains": "Retrieval-Augmented Generation"})

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

### Querying the Collection

We can use the `collection.query()` method to perform a semantic search, using the distance function describe while creating the collection.

We can define the embeddings, texts, images, or ID subsets for similarity search. To limit the results to top n results, use the `n_results` parameter. It also supports filtered queries like the `.get()` method.

In [ ]:
# QUERY: basic semantic search
results = kb.query(
    query_texts=["What is the dental policy?"],
    n_results=3
)
results

{'ids': [['doc_81', 'doc_1128', 'doc_774']],
 'embeddings': None,
 'documents': [['coverage under this Policy commences; \n(10) Dental Treatment - means a treatment related to teeth or structures supporting teeth including examinations, \nfillings (where appropriate), crowns, extractions and surgery',
   'We request you to carefully go through the information given in this document. You are also advised to keep the Policy Bond \nwith utmost care and safety. \n \nYou shall have a period of ___ days from the date of receipt of the Policy Document to review the terms and conditions of this',
   'from \n \nWe request you to carefully go through the information given in this document. You are also advised to \nkeep the Policy Bond with utmost care and safety because this document will be required at the time']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'creator': 'PyPDF',
    'moddate': '2021-12-09T06:23:28+00:00',
    'source': 'd

As you see here, the results of the search contain 3 main components, 'metadatas', 'documents', 'distances'.

In [38]:
results['ids']

[['doc_81', 'doc_1128', 'doc_774']]

In [41]:
results['documents']

[['coverage under this Policy commences; \n(10) Dental Treatment - means a treatment related to teeth or structures supporting teeth including examinations, \nfillings (where appropriate), crowns, extractions and surgery',
  'We request you to carefully go through the information given in this document. You are also advised to keep the Policy Bond \nwith utmost care and safety. \n \nYou shall have a period of ___ days from the date of receipt of the Policy Document to review the terms and conditions of this',
  'from \n \nWe request you to carefully go through the information given in this document. You are also advised to \nkeep the Policy Bond with utmost care and safety because this document will be required at the time']]

We get the distances of each result - lesser distance means higher semantic similarity of result and query. The distances are sorted by closeness.

In [45]:
results['distances']

[[0.26788997650146484, 0.47044336795806885, 0.5068087577819824]]

We can also extract the details from metadata

In [ ]:
# source of the most relevant result
results['metadatas'][0][0]["source"]

'data\\Policy Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf'

In [53]:
# QUERY with metadata filter
results = kb.query(
    query_texts=["What is the dental policy?"],
    n_results=3,
    where_document={"$contains": "dental"}
)
results['documents'], results['distances']

([['7. Routine eye tests, any Dental Treatment or Surgery of cosmetic nature, extraction of impacted \ntooth/teeth, orthodontics or orthognathic surgery, or tempero-mandibular joint disorder except as \nnecessitated by an accidental injury and warranting Hospitalization; \n8. Outpatient treatment;',
   'direct result of the insured event and performed within 6 months of the same). \n• Removal of any material implanted in a former surgery prior to the \ncommencement date. \n• Hospitalisation or surgery in respect of dental treatment of any kind, unless \nnecessitated by accidental bodily injury.',
   '16. Cosmetic or plastic Surgery except to the extent that such Surgery is necessary for the repair of damage \ncaused solely by accidental injuries, cancer or burns; \n17. Treatment of xanthelesema, acne and alopecia; circumcision unless necessary for treatment of a disease']],
 [[0.5198500156402588, 0.5242440700531006, 0.6727714538574219]])

If you see here, the most relevant results are not even related to dental policies. It only considered the documents containing the literal 'dental'. The documents being returned contain it as a substring of the word 'accidental'.

So, we need to write robust queries, and need to prepare and split the documents in such a way that the querying does not break the whole pipeline.

In [54]:
results = kb.query(
    query_texts=["What is the dental policy?"],
    n_results=3,
    where_document={"$contains": "Dental"}
)
results['documents'], results['distances']

([['coverage under this Policy commences; \n(10) Dental Treatment - means a treatment related to teeth or structures supporting teeth including examinations, \nfillings (where appropriate), crowns, extractions and surgery',
   '7. Routine eye tests, any Dental Treatment or Surgery of cosmetic nature, extraction of impacted \ntooth/teeth, orthodontics or orthognathic surgery, or tempero-mandibular joint disorder except as \nnecessitated by an accidental injury and warranting Hospitalization; \n8. Outpatient treatment;']],
 [[0.26788997650146484, 0.5198500156402588]])

You can inspect what is stored in a collection using methods such as `collection.peek()`, and `collection.count()`.

`peek()` is used to return the first few entries. It can also be used for some basic inspection.

In [70]:
# peek
peeked = kb.peek(5)
peeked

{'ids': ['doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5'],
 'embeddings': array([[-5.31179085e-02,  1.33704513e-01,  1.67617481e-02, ...,
          7.98688009e-02, -1.57012977e-02,  1.90880299e-02],
        [-2.89184935e-02,  9.56117436e-02, -3.37499187e-05, ...,
          1.21335462e-02,  3.80633473e-02,  9.15182382e-02],
        [-4.79445346e-02,  9.56941918e-02,  3.95742692e-02, ...,
         -2.05928832e-02, -4.93735820e-02,  1.16626881e-02],
        [ 3.99299748e-02,  1.43205272e-02,  6.00261763e-02, ...,
         -7.96625316e-02, -7.07008988e-02, -1.96314044e-02],
        [-1.04761347e-01,  7.33673051e-02,  6.35947213e-02, ...,
         -5.96840009e-02,  2.05929577e-02, -5.01997694e-02]],
       shape=(5, 384)),
 'documents': ['being this document, has been issued. We have made every effort to design your Policy in a simple format. We \nhave highlighted items of importance so that you may recognize them easily. \n \nPolicy document:',
  'Policy document: \nAs an evidence of the insur

In [ ]:
# Check shape
len(peeked["embeddings"][0])

384

We can also use `.get()` to control which fields are returned.

In [69]:
kb.get(include=["documents", "metadatas"], limit=5)

{'ids': ['doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5'],
 'embeddings': None,
 'documents': ['being this document, has been issued. We have made every effort to design your Policy in a simple format. We \nhave highlighted items of importance so that you may recognize them easily. \n \nPolicy document:',
  'Policy document: \nAs an evidence of the insurance contract between HDFC Life Insurance Company Limited and you, the Policy \nis enclosed herewith. Please preserve this document safely and also inform your nominees about the same. A',
  'copy of your proposal form and other relevant documents submitted by you is also enclosed for your \ninformation and record.  \n \nCancellation in the Free-Look Period: \n \n<< In case you are not agreeable to any of the terms and conditions stated in the Policy, you have the option to',
  'return the Policy to us for cancellation stating the reasons thereof, within 30 days from the date of receipt of the \nPolicy as your Policy is an electronic Policy

We can also count the number of stored items

In [71]:
kb.count()

2372